# AML - TIME TO EVENT - WITH COVARIATES 
This script creates a time-to-event phenotype file for electronic health record–defined acute myeloid leukemia. The earliest electronic health record corresponding to acute myeloid leukemia is considered the event. Time-to-event was calculated both from birth and from the date of the earliest electronic health record. Covariates include documented benzene exposure, smoking status, and exposure to alkylating agents. Additionally, exposure to alkylating agents was assigned only if the electronic health record occurred before the first record of acute myeloid leukemia.

In [ ]:
# Load packages 
from functools import reduce
import pandas as pd 
import os 

In [ ]:
# Set environment for workspace bucket 
my_bucket = os.getenv("WORKSPACE_BUCKET")

In [ ]:
# Download necessary files from workspace bucket 
!gsutil cp {my_bucket}/data/covariates/observation_age_period.tsv .

## AML TIME TO EVENT - ELECTRONIC HEALTH RECORDS  

In [ ]:
import pandas 
import os 

dataset_39009800_person_sql = """
SELECT DISTINCT c.concept_id 
FROM `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
JOIN (
    SELECT CAST(cr.id as string) AS id       
    FROM `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
    WHERE concept_id IN (140352) AND full_text LIKE '%_rank1]%'
) a 
ON (
    c.path LIKE CONCAT('%.', a.id, '.%') 
    OR c.path LIKE CONCAT('%.', a.id) 
    OR c.path LIKE CONCAT(a.id, '.%') 
    OR c.path = a.id
) 
WHERE is_standard = 1 AND is_selectable = 1

UNION DISTINCT 

SELECT DISTINCT concept_id
FROM `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
WHERE concept_id IN (4304199) AND is_standard = 1
"""
aml_concept_ids = pandas.read_gbq(
    dataset_39009800_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

aml_concept_ids.head(5)

# =====================================================================================
# Access electronic health records data for acute myeloid leukaemia 
# =====================================================================================

concept_ids = tuple(aml_concept_ids["concept_id"])

query = f"""                                
SELECT person_id, concept_id, entry_datetime, domain, visit_concept_id, visit_occurrence_id, age_at_event, 
FROM `{os.environ["WORKSPACE_CDR"]}.cb_search_all_events`
WHERE concept_id IN {concept_ids}
"""

aml_phenotype = pd.read_gbq(query, dialect="standard")

## BENZENE EXPOSURE - ELECTRONIC HEALTH RECORDS  

In [ ]:
import pandas
import os

dataset_57070482_person_sql = """
    SELECT
        person.person_id 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person   
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                criteria.person_id 
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                WHERE
                    (concept_id IN (3001306) 
                    AND is_standard = 1  
                    AND  value_as_concept_id IN (45877985) 
                    OR  concept_id IN(SELECT
                        DISTINCT c.concept_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                    JOIN
                        (SELECT
                            CAST(cr.id as string) AS id       
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                        WHERE
                            concept_id IN (3036033, 40776007)       
                            AND full_text LIKE '%_rank1]%'      ) a 
                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                            OR c.path LIKE CONCAT('%.', a.id) 
                            OR c.path LIKE CONCAT(a.id, '.%') 
                            OR c.path = a.id) 
                    WHERE
                        is_standard = 1 
                        AND is_selectable = 1) 
                    AND is_standard = 1 )) criteria 
            UNION
            DISTINCT SELECT
                criteria.person_id 
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                WHERE
                    (concept_id IN(SELECT
                        DISTINCT c.concept_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                    JOIN
                        (SELECT
                            CAST(cr.id as string) AS id       
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                        WHERE
                            concept_id IN (4314382, 40484944, 608234)       
                            AND full_text LIKE '%_rank1]%'      ) a 
                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                            OR c.path LIKE CONCAT('%.', a.id) 
                            OR c.path LIKE CONCAT(a.id, '.%') 
                            OR c.path = a.id) 
                    WHERE
                        is_standard = 1 
                        AND is_selectable = 1) 
                    AND is_standard = 1 )) criteria ) )"""

benzene = pandas.read_gbq(
    dataset_57070482_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

benzene.head(5)

## SMOKING ENDORSEMENT - SELF REPORT AND ELECTRONIC HEALTH RECORDS

In [ ]:
import pandas
import os

dataset_64751293_person_sql = """
    SELECT
        person.person_id 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person   
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                criteria.person_id 
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                WHERE
                    (concept_id IN (1585857) 
                    AND is_standard = 0  
                    AND  value_source_concept_id IN (1585858) 
                    OR  concept_id IN (1586166) 
                    AND is_standard = 0  
                    AND  value_source_concept_id IN (1586167) 
                    OR  concept_id IN (1586174) 
                    AND is_standard = 0  
                    AND  value_source_concept_id IN (1586175) 
                    OR  concept_id IN (1586182) 
                    AND is_standard = 0  
                    AND  value_source_concept_id IN (1586183) 
                    OR  concept_id IN (1586190) 
                    AND is_standard = 0  
                    AND  value_source_concept_id IN (1586191) 
                    OR  concept_id IN(SELECT
                        DISTINCT c.concept_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                    JOIN
                        (SELECT
                            CAST(cr.id as string) AS id       
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                        WHERE
                            concept_id IN (1332802, 1332800)       
                            AND full_text LIKE '%_rank1]%'      ) a 
                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                            OR c.path LIKE CONCAT('%.', a.id) 
                            OR c.path LIKE CONCAT(a.id, '.%') 
                            OR c.path = a.id) 
                    WHERE
                        is_standard = 0 
                        AND is_selectable = 1) 
                    AND is_standard = 0 )) criteria 
            UNION
            DISTINCT SELECT
                criteria.person_id 
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                WHERE
                    (concept_id IN(SELECT
                        DISTINCT c.concept_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                    JOIN
                        (SELECT
                            CAST(cr.id as string) AS id       
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                        WHERE
                            concept_id IN (46273821)       
                            AND full_text LIKE '%_rank1]%'      ) a 
                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                            OR c.path LIKE CONCAT('%.', a.id) 
                            OR c.path LIKE CONCAT(a.id, '.%') 
                            OR c.path = a.id) 
                    WHERE
                        is_standard = 1 
                        AND is_selectable = 1) 
                    AND is_standard = 1 )) criteria 
            UNION
            DISTINCT SELECT
                criteria.person_id 
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                WHERE
                    (concept_id IN (40766360) 
                    AND is_standard = 1  
                    AND  value_as_concept_id IN (1762678) 
                    OR  concept_id IN (43054909) 
                    AND is_standard = 1  
                    AND  value_as_concept_id IN (45884038, 46273821, 45765448, 45883458, 4227477, 903659, 46272634, 44783000, 4021161, 903656, 42709996, 4308357, 4188539, 765320, 4301349, 4276526, 762499, 43054909, 4226233, 36716475, 37395605, 46272595, 40482909, 4219336, 903657, 45878118, 903661, 903652, 4044698, 762498, 4021160, 4011455, 4034855, 40067645, 4052779, 45881517, 4310250, 4298794, 763760, 4191706, 903654) 
                    OR  concept_id IN (4206526) 
                    AND is_standard = 1  
                    AND  value_as_concept_id IN (36715277) 
                    OR  concept_id IN (45890719, 4193014, 4132133, 915749, 40766642, 40766646, 4224317, 4282779, 4090847) 
                    AND is_standard = 1 )) criteria 
            UNION
            DISTINCT SELECT
                criteria.person_id 
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                WHERE
                    (concept_id IN(SELECT
                        DISTINCT c.concept_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                    JOIN
                        (SELECT
                            CAST(cr.id as string) AS id       
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                        WHERE
                            concept_id IN (46273821, 4021160, 44783000)       
                            AND full_text LIKE '%_rank1]%'      ) a 
                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                            OR c.path LIKE CONCAT('%.', a.id) 
                            OR c.path LIKE CONCAT(a.id, '.%') 
                            OR c.path = a.id) 
                    WHERE
                        is_standard = 1 
                        AND is_selectable = 1) 
                    AND is_standard = 1 )) criteria ) )"""

smoking = pandas.read_gbq(
    dataset_64751293_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

smoking.head(5)

## ALKYLATING AGENTS -  ELECTRONIC HEALTH RECORDS 

In [ ]:
import pandas as pd
import os

concept_id_sql = f"""
SELECT DISTINCT
    ca.descendant_id AS concept_id
FROM
    `{os.environ["WORKSPACE_CDR"]}.cb_criteria_ancestor` ca
JOIN
    (SELECT DISTINCT c.concept_id
     FROM `{os.environ["WORKSPACE_CDR"]}.cb_criteria` c
     JOIN 
         (SELECT CAST(cr.id AS STRING) AS id
          FROM `{os.environ["WORKSPACE_CDR"]}.cb_criteria` cr
          WHERE concept_id IN (21601388)
            AND full_text LIKE '%_rank1]%') a
     ON c.path LIKE CONCAT('%.', a.id, '.%')
        OR c.path LIKE CONCAT('%.', a.id)
        OR c.path LIKE CONCAT(a.id, '.%')
        OR c.path = a.id
     WHERE c.is_standard = 1
       AND c.is_selectable = 1) b
ON ca.ancestor_id = b.concept_id
"""

alkylating_agents_concept_ids = pd.read_gbq(
    concept_id_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook"
)

alkylating_agents_concept_ids.head(5)

# =====================================================================================
# Access electronic health records data for alkylating agents 
# =====================================================================================

concept_ids = tuple(alkylating_agents_concept_ids["concept_id"])

query = f"""                                
SELECT person_id, concept_id, entry_datetime, domain, visit_concept_id, visit_occurrence_id, age_at_event, 
FROM `{os.environ["WORKSPACE_CDR"]}.cb_search_all_events`
WHERE concept_id IN {concept_ids}
"""

alkylating_agents = pd.read_gbq(query, dialect="standard")

## CREATE TIME TO EVENT FILE

In [ ]:
# =====================================================================================
# Assign covariate status and select the earliest electronic health record
# =====================================================================================

alkylating_agents["status"] = 1
smoking["status"] = 1
benzene["status"] = 1

aml_phenotype = (
    aml_phenotype
    .sort_values(by=["person_id", "entry_datetime"])
    .drop_duplicates(subset=["person_id"], keep="first")
)

alkylating_agents = (
    alkylating_agents
    .sort_values(by=["person_id", "entry_datetime"])
    .drop_duplicates(subset=["person_id"], keep="first")
)

# =====================================================================================
# Combine the dataframes to create a single consolidated dataframe
# =====================================================================================

aml_phenotype.columns = (
    [aml_phenotype.columns[0]] + ['aml_' + col for col in aml_phenotype.columns[1:]]
)
alkylating_agents.columns = (
    [alkylating_agents.columns[0]] + ['alkylating_agents_' + col for col in alkylating_agents.columns[1:]]
)
smoking.columns = (
    [smoking.columns[0]] + ['smoking_' + col for col in smoking.columns[1:]]
)
benzene.columns = (
    [benzene.columns[0]] + ['benzene_' + col for col in benzene.columns[1:]]
)

dfs = [aml_phenotype,
       alkylating_agents, 
       smoking, 
       benzene]

time_to_event = (
    reduce(lambda left, right: 
           pd.merge(left, right, 
                    on="person_id", 
                    how="outer"), dfs)
)

# =====================================================================================
# Assign missing values a value of zero to indicate nil
# =====================================================================================

time_to_event["smoking_status"] = (
    time_to_event["smoking_status"].fillna(0)
)
time_to_event["benzene_status"] = (
    time_to_event["benzene_status"].fillna(0)
)
time_to_event["alkylating_agents_status"] = (
    time_to_event["alkylating_agents_status"].fillna(0)
)

time_to_event['aml_entry_datetime'] = (
    pd.to_datetime(time_to_event['aml_entry_datetime'])
)
time_to_event['alkylating_agents_entry_datetime'] = (
    pd.to_datetime(time_to_event['alkylating_agents_entry_datetime'])
)

# =====================================================================================
# Calculate time to event and assign event status 
# =====================================================================================

observation_age_period = pd.read_csv("observation_age_period.tsv", sep ='\t')

time_to_event["event"] = (
    time_to_event["aml_entry_datetime"].dt.date
)

time_to_event["alkylating_agents_event"] = (
    time_to_event["alkylating_agents_entry_datetime"].dt.date
)

time_to_event = (
    time_to_event.merge(observation_age_period, 
                        on ="person_id", 
                        how = "outer")
)

time_to_event["status"] = (
    time_to_event["event"].apply(lambda x: 1 if pd.isna(x) else 2)
)

time_to_event["event"] = (
    time_to_event["event"]
    .fillna(time_to_event["observation_period_end_date"])
)

time_to_event["observation_period_start_date"] = (
    pd.to_datetime(time_to_event["observation_period_start_date"])
)

time_to_event["event"] = (
    pd.to_datetime(time_to_event["event"])
)

time_to_event["observation_period_start_date"] = (
    pd.to_datetime(time_to_event["observation_period_start_date"])
)

time_to_event["date_of_birth"] = (
    pd.to_datetime(time_to_event["date_of_birth"])
)

time_to_event["time_to_event_days_from_observation"] = (
    time_to_event["event"] - time_to_event["observation_period_start_date"]
).dt.days

time_to_event["time_to_event_days_from_birth"] = (
    time_to_event["event"] - time_to_event["date_of_birth"]
).dt.days

time_to_event["age_at_observation_period_start"] = (
    (time_to_event["observation_period_start_date"] - time_to_event["date_of_birth"]
).dt.days / 365.25)

time_to_event["observation_period_start_year"] = (
        time_to_event["observation_period_start_date"].dt.year
)

time_to_event["date_of_birth_year"] = (
    time_to_event["date_of_birth"].dt.year
)

# =====================================================================================
# Calculate time dependent covariate status 
# =====================================================================================

time_to_event["alkylating_agents_status"] = (
    time_to_event["alkylating_agents_entry_datetime"]
    .apply(lambda x: 1 if pd.isna(x) else 2)
)

time_to_event["alkylating_agents_entry_datetime"] = (
    time_to_event["alkylating_agents_entry_datetime"].dt.tz_localize(None)
)

time_to_event["event"] = (
    time_to_event["event"].dt.tz_localize(None)
)

time_to_event["alkylating_agents_entry_datetime"] = (
    time_to_event["alkylating_agents_entry_datetime"]
    .fillna(time_to_event["event"])
)

time_to_event["alkylating_agents_days_from_event"] = (
    time_to_event["event"] - time_to_event["alkylating_agents_entry_datetime"]
).dt.days

time_to_event.loc[
    time_to_event["alkylating_agents_days_from_event"] > 1095,
    "alkylating_agents_status_time_3years"
] = 1


time_to_event.loc[
    time_to_event["alkylating_agents_days_from_event"] > 0,
    "alkylating_agents_status_time"
] = 1

time_to_event["alkylating_agents_status_time_3years"] = (
    time_to_event["alkylating_agents_status_time_3years"].fillna(0)
)

time_to_event["alkylating_agents_status_time"] = (
    time_to_event["alkylating_agents_status_time"].fillna(0)
)

time_to_event["alkylating_agents_status_time"] = (
    time_to_event["alkylating_agents_status_time"].fillna(0)
)

# =====================================================================================
# Save time to event with covariates file 
# =====================================================================================

time_to_event.to_csv("aml_phenotype_time_to_event_with_covariates.tsv", 
                     sep ='\t', 
                     index = None)

In [ ]:
# Save the file to the workspace bucket 
!gsutil cp aml_phenotype_time_to_event_with_covariates.tsv {my_bucket}/data/phenotype_files/ 